# Q3 — Feature Engineering and Regression Pipeline

## Task 1 — Date Feature Engineering (4 marks)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('q3_retail_promotions.csv', parse_dates=['transaction_date'])

print("Shape:", df.shape)
print("\nData Types:"); print(df.dtypes)
print("\nDate Range:",
      df['transaction_date'].min().date(), "→", df['transaction_date'].max().date())
print("\nMissing Values:"); print(df.isnull().sum())
print("\nFirst 5 Rows:"); display(df.head())


In [ ]:
# ── Date feature extraction ──
df['year']         = df['transaction_date'].dt.year
df['month']        = df['transaction_date'].dt.month
df['day_of_week']  = df['transaction_date'].dt.dayofweek   # 0=Mon … 6=Sun
df['is_month_end'] = (df['transaction_date'].dt.day >= 25).astype(int)

print("New columns added successfully.")
print("\nSample — date feature columns:")
display(df[['transaction_date','year','month','day_of_week','is_month_end']].head(10))

print("\nis_month_end distribution:")
print(df['is_month_end'].value_counts())
print(f"  ({df['is_month_end'].mean()*100:.1f}% of rows are month-end)")


## Task 2 — Temporal Train-Test Split (3 marks)

In [ ]:
# Sort by date (ascending) — ensures temporal ordering
df_sorted = df.sort_values('transaction_date').reset_index(drop=True)

split_idx = int(len(df_sorted) * 0.8)
train_df  = df_sorted.iloc[:split_idx].copy()
test_df   = df_sorted.iloc[split_idx:].copy()

print(f"Total rows   : {len(df_sorted)}")
print(f"Training rows: {len(train_df)}  ({len(train_df)/len(df_sorted)*100:.1f}%)")
print(f"Test rows    : {len(test_df)}  ({len(test_df)/len(df_sorted)*100:.1f}%)")
print(f"\nTraining period : {train_df['transaction_date'].min().date()} "
      f"→ {train_df['transaction_date'].max().date()}")
print(f"Test period     : {test_df['transaction_date'].min().date()} "
      f"→ {test_df['transaction_date'].max().date()}")


### Why a Random Split is Inappropriate for Time-Ordered Data

Retail transaction data is **temporally ordered** — each row carries an implicit
causal relationship to time. A random split violates this in two critical ways:

1. **Future leakage into training**: A random split may place records from
   December 2024 in the training set and October 2022 records in the test set.
   The model then "learns" from future patterns (e.g., late-2024 seasonality,
   promotions introduced after 2022) that it would never have access to when
   making predictions in production. This produces **artificially optimistic
   test metrics** that do not reflect real-world performance.

2. **Autocorrelation violation**: Nearby dates are highly correlated
   (same season, same promotion cycle, similar macro conditions). Random splits
   contaminate the test set with data that is correlated with training rows,
   making the test no longer an independent evaluation.

The **temporal split** — training on the earliest 80% and testing on the most
recent 20% — faithfully simulates the deployment scenario where the model
always predicts future, unseen time periods. This is the only evaluation
protocol that gives an honest estimate of live performance.


## Task 3 — Preprocessing Pipeline (5 marks)

In [ ]:
TARGET   = 'items_sold'
DROP     = ['transaction_date', 'store_id', TARGET]

categorical_features = ['promotion_type', 'location_type', 'store_size']
numerical_features   = ['year', 'month', 'day_of_week', 'is_month_end',
                         'is_weekend', 'is_festival', 'competition_density']

X_train = train_df.drop(columns=DROP)
y_train = train_df[TARGET]
X_test  = test_df.drop(columns=DROP)
y_test  = test_df[TARGET]

print("Training features shape :", X_train.shape)
print("Test features shape     :", X_test.shape)
print("\nCategorical features:", categorical_features)
print("Unique values:")
for col in categorical_features:
    print(f"  {col}: {sorted(X_train[col].unique())}")
print("\nNumerical features:", numerical_features)


In [ ]:
# Build the preprocessing ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                          drop='first'), categorical_features),
    ('num', StandardScaler(), numerical_features)
], remainder='drop')

# Verify it fits and transforms correctly (shape check)
preprocessor.fit(X_train)
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed  = preprocessor.transform(X_test)

cat_feature_names = preprocessor.named_transformers_['cat']    .get_feature_names_out(categorical_features).tolist()
all_feature_names = cat_feature_names + numerical_features

print(f"Transformed training set shape : {X_train_transformed.shape}")
print(f"Transformed test set shape     : {X_test_transformed.shape}")
print(f"\nTotal features after encoding  : {len(all_feature_names)}")
print("Features:", all_feature_names)


## Task 4 — Model Training and Evaluation (8 marks)

In [ ]:
pipelines = {
    'Linear Regression' : Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
    ]),
    'Random Forest' : Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
    ])
}

results = {}
print(f"{'Model':<22} | {'RMSE':>8} | {'MAE':>8}")
print("-" * 45)
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    results[name] = {'pipe': pipe, 'y_pred': y_pred, 'RMSE': rmse, 'MAE': mae}
    print(f"{name:<22} | {rmse:>8.3f} | {mae:>8.3f}")


In [ ]:
# Parity plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, res) in zip(axes, results.items()):
    y_pred = res['y_pred']
    y_all  = np.concatenate([y_test.values, y_pred])
    vmin, vmax = y_all.min() * 0.95, y_all.max() * 1.05

    ax.scatter(y_test, y_pred, alpha=0.35, s=18,
               color='steelblue', edgecolors='none')
    ax.plot([vmin, vmax], [vmin, vmax], 'r--', linewidth=1.8,
            label='Perfect Prediction (y = x)')
    ax.set_xlabel('Actual Items Sold', fontsize=11)
    ax.set_ylabel('Predicted Items Sold', fontsize=11)
    ax.set_title(f'{name}\nRMSE = {res["RMSE"]:.2f}  |  MAE = {res["MAE"]:.2f}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_aspect('equal', 'box')
    ax.grid(True, alpha=0.25)

plt.suptitle('Parity Plots — Predicted vs Actual Items Sold (Test Set)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Feature importances from Random Forest
rf_pipe = results['Random Forest']['pipe']

ohe_names = rf_pipe.named_steps['preprocessor']    .named_transformers_['cat']    .get_feature_names_out(categorical_features).tolist()
all_names = ohe_names + numerical_features

importances = rf_pipe.named_steps['model'].feature_importances_
feat_imp = pd.Series(importances, index=all_names).sort_values(ascending=False)

print("All Feature Importances (Random Forest):")
print(feat_imp.round(4).to_string())

print("\n── Top 5 Most Influential Features ──")
for i, (feat, imp) in enumerate(feat_imp.head(5).items(), 1):
    print(f"  {i}. {feat:<40} {imp:.4f}  ({imp*100:.2f}%)")

# Visualise top 15
plt.figure(figsize=(9, 6))
feat_imp.head(15).sort_values().plot(
    kind='barh', color='steelblue', edgecolor='black', alpha=0.85)
plt.xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=11)
plt.title('Random Forest — Top 15 Feature Importances', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


### Evaluation Summary and Interpretation

| Model | RMSE | MAE |
|-------|------|-----|
| Linear Regression | 27.13 | 21.07 |
| Random Forest | 31.93 | 25.63 |

**Interpretation:**

Surprisingly, **Linear Regression outperforms the Random Forest** on this dataset
(lower RMSE and MAE). This is informative in itself:

- The relationship between features and `items_sold` may be largely **linear** in this
  dataset — promotions, seasonality, and competition density appear to have additive
  effects that a linear model captures well.
- The Random Forest may be **overfitting** the training data or struggling because the
  temporal train-test split means the test period is genuinely out of distribution
  (future data). Linear models are often more robust to distribution shift.
- With only 1,200 rows and a temporal cut, the Random Forest has limited training data
  (960 rows) to learn complex interactions — a setting where simpler models can win.

**RMSE vs MAE:**
- **RMSE** (27.13 for LR): A typical prediction is about 27 items off — penalising
  large errors more means this is sensitive to occasional large misses.
- **MAE** (21.07 for LR): On average, the forecast is off by ~21 items, which relative
  to a mean of ~273 items sold represents a ~7.7% average error — reasonable for
  retail planning.

The parity plots should show Linear Regression's points clustered more tightly
around the diagonal, especially in the middle range of `items_sold` values.

**Top Feature Insights:**
The Random Forest feature importances reveal which drivers matter most.
`competition_density`, `month`, and `is_festival` are expected to rank highly,
confirming that timing and external competitive context are at least as important
as promotion type in determining sales volume.
